# E1 — CV 5-fold por paciente (Etapa B)

Protocolo (seção 2.3): para cada fold *k*, treina nos outros quatro folds do desenvolvimento, escolhe
o checkpoint pela validação fixa e avalia **só no fold k**. Os hiperparâmetros vêm congelados de
`configs/frozen/e1-hparams.yaml`. O teste INCA não é carregado.

Agregação (seção 4): **paciente → fold**, ou seja média dos frames de cada paciente, média dos pacientes do
fold, e depois média ± desvio-padrão entre os 5 folds. As comparações entre arquiteturas são **diferenças
pareadas por fold**. O desvio entre folds mede sensibilidade à partição: os folds compartilham pacientes
de treino e não são repetições independentes.

ASSD/HD95 estão em pixels a 256×256.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../../../")

import matplotlib.pyplot as plt
import pandas as pd
import yaml

from src.data.splits import resolve_project_path
from src.evaluation import METRIC_NAMES, aggregate_per_patient, paired_fold_differences, summarize_over_folds
from src.sweep import Sweep, submit_sweep, sweep_status, watch_sweep

SWEEP_CONFIG = "configs/sweep/e1-cv.yaml"
REFERENCE_ARCH = "unet"

sweep = Sweep(SWEEP_CONFIG)
frozen = yaml.safe_load(resolve_project_path(sweep.config.hparams_file).read_text())
pd.DataFrame(frozen["hparams"]).T

## Execução

In [ ]:
sweep_status(SWEEP_CONFIG)

In [ ]:
# submit_sweep(SWEEP_CONFIG)
# watch_sweep(SWEEP_CONFIG)

## Por fold (paciente → fold)

In [ ]:
results = sweep.load_results()
assert set(results.eval_set) == {"cv"}, "o CV so pode ter linhas do fold retido"

per_fold = aggregate_per_patient(results, group_columns=["arch", "fold", "seed"])
per_fold.pivot_table(index="fold", columns="arch", values="dice").round(4)

## Média ± desvio-padrão entre folds

In [ ]:
summary = summarize_over_folds(per_fold, ["arch"])
table = pd.DataFrame({"arch": summary.arch, "n_folds": summary.n_folds})
for metric in METRIC_NAMES:
    table[metric] = [f"{m:.4f} ± {s:.4f}" for m, s in zip(summary[f"{metric}_mean"], summary[f"{metric}_std"])]
table

## Diferenças pareadas por fold contra a U-Net

Positivo = maior que a referência (melhor para Dice/IoU, pior para ASSD/HD95).

In [ ]:
diffs = paired_fold_differences(per_fold, by="arch", reference=REFERENCE_ARCH, group_columns=["seed"])
display(diffs.pivot_table(index="fold", columns="arch", values="dice").round(4))
summarize_over_folds(diffs, ["arch"])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for arch, group in per_fold.groupby("arch"):
    group = group.sort_values("fold")
    ax.plot(group.fold, group.dice, marker="o", label=arch)
ax.set_xlabel("fold"); ax.set_ylabel("Dice médio por paciente"); ax.set_xticks(range(5))
ax.legend(); ax.grid(alpha=0.3)